# SWAP PCA RV TOOL

In [ ]:
import pdblp
import matplotlib.pyplot as plt
from typing import Iterable, List, Tuple, Union, Optional, Dict

import datetime as dt
import pandas as pd
import numpy as np
import statsmodels.api as sm

from pandas.tseries.offsets import DateOffset

import swap_pca_screener
import InteractivePlotter
import importlib


importlib.reload(swap_pca_screener)
importlib.reload(InteractivePlotter)

from swap_pca_screener import PCAScreenerConfig, SwapCurvePCAScreener


## BBG Connection

In [1]:
con = pdblp.BCon(debug=False, port=8194, timeout=50000)


NameError: name 'pdblp' is not defined

In [ ]:
con.start()


## Functions:

In [3]:
########################
# BBG Download
########################

def bg_download_clean(
    ticker_list,
    fields,
    start_date,
    end_date,
    period: Optional[str] = None
):
    start_date = start_date.strftime("%Y%m%d")
    end_date = end_date.strftime("%Y%m%d")

    if period is None:
        tmp = con.bdh(ticker_list, fields, start_date, end_date)
    else:
        tmp = con.bdh(
            ticker_list,
            fields,
            start_date,
            end_date,
            elms=[('periodicitySelection', period)]
        )

    tmp.columns = tmp.columns.droplevel(1)
    tmp = tmp.sort_values(by="date", ascending=False)
    return tmp


def bg_ticker(ticker_list, fld):
    bg_df = con.bdp(ticker_list, fld)
    bg_df.columns = [fld]
    bg_df.index = ticker_list
    return bg_df


def make_bg_ticker(
    root: str,
    tenor: str,
    source: str = "BLCS",
    asset: str = "Currency"
) -> str:
    return f"{root} {tenor} {source} {asset}"

def normalize_tenor(tenor: Union[str, float, int]) -> str:
    if isinstance(tenor, float):
        return str(int(tenor))
    if isinstance(tenor, int):
        return str(tenor)

    t = tenor.strip().upper()
    t = t.replace(" ", "")
    t = t.replace("YRS", "Y")
    t = t.replace("YR", "Y")

    if not t.endswith("Y"):
        raise ValueError(f"Tenor not valid: {tenor}. Use e.g. '1Y','2Y','10Y'.")

    return t


def generate_bg_tickers(
    tenors: Union[List[int], List[str]],
    root: str = "SOFR",
    source: str = "BLCS",
    asset: str = "Currency"
):
    out = []

    if isinstance(tenors, list):
        tenors = [normalize_tenor(t) for t in tenors]

    for t in tenors:
        ticker = make_bg_ticker(root, t, source, asset)
        out.append(ticker)

    return out

def forward_series_from_timeseries(
    df,
    start_tenor,
    end_tenor,
    compounding: str = "annual"
):

    def _parse_tenor_to_years(s):
        s = str(s).upper().strip()
        if s.endswith("M"):
            return float(s[:-1]) / 12.0
        elif s.endswith("Y"):
            return float(s[:-1])
        else:
            return float(s)

    T1 = _parse_tenor_to_years(start_tenor)
    T2 = _parse_tenor_to_years(end_tenor)

    tenors = [c for c in df.columns if c != "date"]
    tenor_years = [_parse_tenor_to_years(t) for t in tenors]

    sort_idx = np.argsort(tenor_years)
    tenors = [tenors[i] for i in sort_idx]
    tenor_years = [tenor_years[i] for i in sort_idx]

    results = []

    for _, row in df.iterrows():
        curve = row[tenors].values.astype(float)

        # Interpolazione dei tassi spot per T1 e T2
        R1 = np.interp(T1, tenor_years, curve)
        R2 = np.interp(T2, tenor_years, curve)

        if compounding == "simple":
            fwd = ((1 + R2 * T2) / (1 + R1 * T1) - 1) / (T2 - T1)
        else:
            fwd = ((1 + R2) ** T2 / (1 + R1) ** T1) ** (1 / (T2 - T1)) - 1

        results.append(fwd)

    out = pd.DataFrame(
        {
            "Date": df.index,
            f"Fwd_{str(int(T1))}Y_{str(int(T2 - T1))}Y": results,
        }
    )

    return out.set_index("Date")

def interpolate_curve(df, new_tenors):

    def _to_years(t):
        t = t.upper().strip()
        if t.endswith("M"):
            return float(t[:-1]) / 12
        elif t.endswith("Y"):
            return float(t[:-1])
        else:
            return float(t)

    # Converte colonne in anni e ordinale
    tenor_years = np.array([_to_years(c) for c in df.columns])
    sort_idx = np.argsort(tenor_years)

    df = df.iloc[:, sort_idx]
    tenor_years = tenor_years[sort_idx]

    # Converte i nuovi tenori
    new_years = np.array([_to_years(t) for t in new_tenors])

    # Interpolazione per ogni data
    interpolated_data = {}

    for new_y, new_t in zip(new_years, new_tenors):
        interpolated_data[new_t] = [
            np.interp(new_y, tenor_years, row.values)
            for _, row in df.iterrows()
        ]

    # Combino tutto in un unico DataFrame
    interpolated_df = pd.DataFrame(interpolated_data, index=df.index)
    full_df = pd.concat([df, interpolated_df], axis=1)

    # Ordino le colonne in base alla maturità
    full_df = full_df[sorted(full_df.columns, key=_to_years)]

    return full_df

def generate_curve_spreads(df, tenor1, tenor2, fwd=""):
    if fwd != "":
        tmp = pd.DataFrame(100 * (df[fwd + tenor2] - df[fwd + tenor1]))
        tmp.columns = [fwd + tenor2 + "-" + tenor1]
        return tmp
    else:
        tmp = pd.DataFrame(100 * (df[tenor2] - df[tenor1]))
        tmp.columns = [tenor2 + "-" + tenor1]
        return tmp


def generate_butterflies(
    df: pd.DataFrame,
    belly: str,
    wing1: str,
    wing2: str,
    fwd: str = "",
    w_belly: float = 2.0,
    w_wing1: float = -1.0,
    w_wing2: float = -1.0,
    scale: float = 100.0,
    colname: str = None,
) -> pd.DataFrame:

    if fwd != "":
        col_belly = fwd + belly
        col_w1 = fwd + wing1
        col_w2 = fwd + wing2
    else:
        col_belly = belly
        col_w1 = wing1
        col_w2 = wing2

    fly_series = scale * (
        w_belly * df[col_belly]
        + w_wing1 * df[col_w1]
        + w_wing2 * df[col_w2]
    )

    if colname is None:
        colname = f"{col_w1}_{col_belly}_{col_w2}"

    tmp = pd.DataFrame(fly_series)
    tmp.columns = [colname]
    return tmp



NameError: name 'Optional' is not defined

### Data Download:

In [2]:
tenors_integers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 15, 20, 25, 30, 50]
user_tenors = [str(t) + "Y" for t in tenors_integers]

tickers = generate_bg_tickers(user_tenors, root = "S0045P", source = "BLC3", asset = "Curncy")


from_ = dt.datetime(2020, 1, 1)
to_ = dt.datetime.today()
flds = "PX_LAST"


data = bg_downlaod_clean(tickers, flds, from_, to_)
data.columns = user_tenors
data = data.dropna()

data.tail()


_IncompleteInputError: incomplete input (3700930997.py, line 2)

### Interpolate Missing Maturities:

In [ ]:
# Interpolate missing maturities

all_t = [f"{i}Y" for i in range(1, 51)]

# Tenors to interpolate
to_interp = [t for t in all_t if t not in tenors_integers]
to_interp_str = [str(t) for t in to_interp]

data_swap_full_tenors = interpolate_curve(data, to_interp_str)
data_swap_full_tenors_pct = data_swap_full_tenors / 100

data_swap_full_tenors_pct.head()

## Computing Fwd rates:

In [ ]:
# Used fwd: 1y1y, 2y1y, 3y1y, 4y1y, 5y1y, 6y1y, 7y1y, 8y1y, 9y1y, 10y1y,
#           1y2y, 1y3y, 1y5y, 2y5y, 3y5y, 5y5y, 10y5y, 10y10y

fwd_rates_tuples = [
    ("1Y", "2Y"),
    ("2Y", "3Y"),
    ("3Y", "4Y"),
    ("4Y", "5Y"),
    ("5Y", "6Y"),
    ("6Y", "7Y"),
    ("7Y", "8Y"),
    ("8Y", "9Y"),
    ("9Y", "10Y"),
    ("10Y", "12Y"),
    ("12Y", "15Y"),
    ("15Y", "20Y"),
    ("20Y", "25Y"),
    ("25Y", "30Y"),
    ("30Y", "40Y"),
    ("40Y", "50Y")
]

# Initialize DataFrame
fwd_swaps_df = pd.DataFrame()

for i in fwd_rates_tuples:
    tmp_df = forward_series_from_timeseries(
        data_swap_full_tenors_pct,
        i[0],
        i[-1],
        compounding="annual",
    )

    fwd_swaps_df = pd.concat([fwd_swaps_df, tmp_df], axis=1)

fwd_swaps_df = fwd_swaps_df * 100
fwd_swaps_df = pd.concat([data[["1Y"]], fwd_swaps_df], axis=1)


## Fwd Curve as of Today:

In [ ]:
plotter_fwd = InteractivePlotter.InteractivePlotter(df=fwd_swaps_df)

last_point = fwd_swaps_df.iloc[1]
w1_ago = fwd_swaps_df.iloc[5]
m1_ago = fwd_swaps_df.iloc[22]

curve_evolution = pd.concat(
    [last_point, w1_ago, m1_ago],
    axis=1
)

plotter_fwd.bar(
    curve_evolution,
    title="curve fwd (vs.)",
    mode="group"
)

In [ ]:
w_chg = pd.DataFrame(100*(last_point - w1_ago))

plotter_chg = InteractivePlotter.InteractivePlotter(df=w_chg)
plotter_chg.bar(
    w_chg,
    title="fwd changes 1w",
    mode="group"
)


# Block 1: Performing PCA

### PCA residuals as of today:

In [ ]:
# Inputs
roll_period = 250
n_pcs = 2

# Initializing Class
config = PCAScreenerConfig(
    n_components=n_pcs,
    window_mode="rolling",
    rolling_window=roll_period,
    use_changes = False
    standardize_by_vol=True,
)

screener = SwapCurvePCAScreener(config)
screener.fit(fwd_swaps_df)

res_pca_score = screener.get_signals()
res_pca_res = screener.get_residuals()




#### Z-scores:

In [ ]:
plotter = InteractivePlotter.InteractivePlotter(
    df=res_pca_score,
    template="plotly_white"
)

last_residuals = res_pca_score.iloc[-1]

plotter.bar(
    last_residuals,
    title="last pca residuals Zscore (vs 6)"
)


### In bp

In [ ]:
res_pca_bp = res_pca_bp *100

plotter = InteractivePlotter.InteractivePlotter(df = res_pca_bp, template = "ploty_white")
last_residuals = res_pca_bp.iloc[-1]

plotter.bar(
    last_residuals,
    title="last pca residuals Zscore (vs 6)"
)

### Inspecting time series of z-score residuals for a given tenor:

In [ ]:
to_plot = res_pca_z_score[["Fwd_4Y_1Y"]]

plotter_res = InteractivePlotter.InteractivePlotter(df = to_plot)
plotter_res.timeseries(to_plot.columns[0], title = "Z-score timeseries")


# BLock 2: Evolution of residuals at different point in time:

In [ ]:
def get_zscores_at_offsets(
    screener: SwapCurvePCAScreener,
    as_of: Optional[pd.Timestamp] = None,
    offsets: Optional[Dict[str, Dict[str, int]]] = None,
) -> pd.DataFrame:

    signals = screener.get_signals()
    if signals.empty:
        raise RuntimeError("No signals available; did you call fit()?")

    # choose reference date
    if as_of is None:
        ref_date = signals.index.max()
    else:
        ref_date = pd.Timestamp(as_of)

    # default offsets
    if offsets is None:
        offsets = {
            "today": {"days": 0},
            "1w": {"days": 7},
            "1m": {"months": 1},
            "3m": {"months": 3},
        }

    # helper: nearest index <= target_date
    def nearest_on_or_before(
        target: pd.Timestamp,
        idx: pd.DatetimeIndex
    ) -> Optional[pd.Timestamp]:
        valid = idx[idx <= target]
        if len(valid) == 0:
            return None
        return valid.max()

    idx = signals.index
    selected_rows = {}

    for label, off in offsets.items():
        target = ref_date
        if "days" in off:
            target = target - pd.Timedelta(days=off["days"])
        if "months" in off:
            target = target - DateOffset(months=off["months"])

        actual = nearest_on_or_before(target, idx)
        if actual is None:
            continue

        selected_rows[label] = signals.loc[actual]

    if not selected_rows:
        raise ValueError("No matching dates found for the given offsets.")

    out = pd.DataFrame(selected_rows)
    out.index.name = "tenor"
    return out


### Initializing PCA

In [ ]:
### Initializing PCA.

config = PCAScreenerConfig(
    n_components=3,
    window_mode="rolling",
    rolling_window=250,
    use_changes=False,
    standardize_by_vol=True,
    min_window_obs=65,
)

screener = SwapCurvePCAScreener(config).fit(fwd_swaps_df)


#### Exctract Residuals:

In [ ]:
### Extract residuals

custom_offsets = {
    "today": {"days": 0},
    "1w": {"days": 7},
    "1m": {"days": 30},
    "3m": {"days": 90},
}

zsnap_custom = get_zscores_at_offsets(screener, offsets=custom_offsets)
plotter = InteractivePlotter.InteractivePlotter(df=zsnap_custom, template="plotly_white")

plotter.bar(zsnap_custom, title="Residuals z-scores evolutions: Rolling PCA (1y roll)")


# Block 3: PCA Comparison using different Samples:

In [ ]:
def run_multi_horizon_pca(
    data: pd.DataFrame,
    n_components: int = 3,
    use_changes: bool = False,
    standardize_by_vol: bool = True,
    trading_days_per_year: int = 252,
) -> Tuple[Dict[str, SwapCurvePCAScreener], Dict[str, pd.DataFrame]]:
    """
    Run the PCA screener on multiple horizons:
        - full sample
        - ~2y rolling
        - ~1y rolling
        - ~6m rolling
        - ~3m rolling

    Returns
    -------
    screeners : dict
        key = horizon label, value = fitted SwapCurvePCAScreener.
    signals : dict
        key = horizon label, value = DataFrame of z-score residuals
        (date x tenor) for that horizon.
    """
    horizons = {
        "full": None,
        "2y": 2 * trading_days_per_year,
        "1y": 1 * trading_days_per_year,
        "6m": trading_days_per_year // 2,
        "3m": trading_days_per_year // 4,
    }

    screeners: Dict[str, SwapCurvePCAScreener] = {}
    signals: Dict[str, pd.DataFrame] = {}

    for name, window in horizons.items():
        if name == "full":
            cfg = PCAScreenerConfig(
                n_components=n_components,
                window_mode="full_sample",
                use_changes=use_changes,
                change_lag=1,
                standardize_by_vol=standardize_by_vol,
                # rolling_window/min_window_obs irrelevant in full_sample
            )
        else:
            cfg = PCAScreenerConfig(
                n_components=n_components,
                window_mode="rolling",
                rolling_window=int(window),
                use_changes=use_changes,
                change_lag=1,
                standardize_by_vol=standardize_by_vol,
                min_non_nan=0.95,
                min_window_obs=int(window),
            )

        scr = SwapCurvePCAScreener(cfg).fit(data)
        screeners[name] = scr
        signals[name] = scr.get_signals()

    return screeners, signals

def compare_zscores_at_date(
    signals_dict: Dict[str, pd.DataFrame],
    as_of: Optional[pd.Timestamp] = None,
) -> pd.DataFrame:
    """
    For a given date, compare z-score residuals across horizons.

    Parameters
    ----------
    signals_dict : dict
        key = horizon label (e.g. "full","2y"), value = DataFrame
              of z-score residuals (date x tenor).
    as_of : pd.Timestamp or None
        If None, use the last common date across all horizons.

    Returns
    -------
    DataFrame
        index = tenors
        columns = horizon labels
        values = z-scores at 'as_of' for each (tenor, horizon).
    """
    # compute intersection of dates across all horizons
    common_dates = None
    for _, df in signals_dict.items():
        idx = set(df.index)
        if common_dates is None:
            common_dates = idx
        else:
            common_dates = common_dates & idx

    if not common_dates:
        raise ValueError("No common dates across horizons.")

    if as_of is None:
        as_of = max(common_dates)
    else:
        as_of = pd.Timestamp(as_of)
        if as_of not in common_dates:
            raise ValueError(f"Date ({as_of}) is not common to all horizons.")

    rows = {}
    for name, df in signals_dict.items():
        rows[name] = df.loc[as_of]

    # index = tenor, columns = horizon (full,2y,1y,6m,3m)
    comp = pd.DataFrame(rows)
    comp.index.name = "tenor"
    return comp


### Results:

In [ ]:
screeners, signals = run_multi_horizon_pca(
    fwd_swaps_df,
    n_components=2,
    use_changes=False,
    standardize_by_vol=True,
)


comp = compare_zscores_at_date(signals)


In [ ]:
plotter = InteractivePlotter.InteractivePlotter(
    df=comp,
    template="plotly_white"
)

plotter.bar(
    comp,
    title="Residuals z-scores as of today using different rolling periods: 2 PCs"
)


# Block 4: Trade Screener

### Curve Screener:

In [ ]:
col_order = list(fwd_swaps_df.columns)

curve_resid = screener_df.screen_curve_spreads_from_residuals(
    as_of=None,
    tenor_order=col_order,
    min_history=250,
    only_adjacent_pairs=True,
    n=30,
    abs_threshold=1.5,
)


In [ ]:
curve_resid

### Flies Screener

In [ ]:
flies_resid = screener_df.screen_flies_from_residuals(
    as_of=None,
    tenor_order=col_order,
    min_history=250,
    n=30,
    abs_threshold=0.5,
    belly_gap=2,
)


In [ ]:
flies_resid


### Inspection:


In [ ]:
fly_ = generate_butterflies(
    fwd_swaps_df,
    "Fwd_2Y_1Y",
    "Fwd_4Y_1Y",
    "Fwd_7Y_1Y",
)

fly_res = generate_butterflies(
    res_pca_bp/100,
    "Fwd_2Y_1Y",
    "Fwd_4Y_1Y",
    "Fwd_7Y_1Y",
)
    

In [ ]:
plot = InteractivePlotter.InteractivePlotter(df = fly_)
plot.timeseries(fly_.columns[0])

In [ ]:
plot = InteractivePlotter.InteractivePlotter(df = fly_res)
plot.timeseries(fly_.columns[0], title = " fly bp res")

# Block 5: PCA Weighted fly and directionality :

In [ ]:
config = PCAScreenerConfig(
    n_components=2,
    window_mode="rolling",
    rolling_window=250,
    use_changes=False,
    standardize_by_vol=True,
)

screener = SwapCurvePCAScreener(config).fit(fwd_swaps_df)
as_of = screener.rolling_loadings_.index.get_level_values("date").max()

# roll = screener.get_pca_weights(
#     as_of=as_of,
#     short_tenor="Fwd_2Y_1Y",
#     belly_tenor="Fwd_4Y_1Y",
#     long_tenor="Fwd_7Y_1Y",
#     pc=0,
#     normalize_belly=True,
# )

w_roll = screener.get_pca_weights(
    as_of=as_of,
    short_tenor="Fwd_2Y_1Y",
    belly_tenor="Fwd_4Y_1Y",
    long_tenor="Fwd_7Y_1Y",
    pc=0,
    normalize_belly=True,
)

w_roll



In [ ]:
pca_weighted_fly = generate_butterflies(
    data=fwd_swaps_df,
    belly="Fwd_4Y_1Y",
    wing1="Fwd_2Y_1Y",
    wing2="Fwd_7Y_1Y",
    w_belly=1.0,
    w_wing1=roll["Fwd_2Y_1Y"],
    w_wing2=roll["Fwd_7Y_1Y"],
    scale=100,
)


In [ ]:
res = screener.run_ols_regression_with_plots(
    df=pca_check_directionality,
    x_col="Fwd_2Y_1Y",
    y_col=pca_weighted_fly.columns[0],
)
